# Synthetic EEG Pipeline — DBS Artifact Removal + ICA

End-to-end pipeline on a 19-channel (10-20) synthetic signal containing:
- **7 Hz DBS** artifact (fundamental + 17 harmonics up to Nyquist)
- **Eye movement** artifacts (blinks + horizontal saccades on frontal channels)
- **Muscle (EMG)** artifacts on temporal / outer-frontal channels

## Why the comb-notch Q=50 was over-aggressive

A `scipy.signal.iirnotch` with Q=50 applied via `filtfilt` (zero-phase, doubles the
filter order) has a **−1 dB bandwidth of ≈ 0.39 Hz** at every harmonic — not 0.14 Hz
as the formula f₀/Q would suggest.  With 7 Hz DBS harmonics at 14, 21, 28 Hz all
inside the beta band, cascading 0.39 Hz notches across those frequencies removes
~9% of brain beta power.

## Solution: FFT Spectral Interpolation

For a 120 s recording at 256 Hz the FFT frequency resolution is **0.0083 Hz/bin**.
Our FFT spectral interpolation removes **±2 bins = 0.042 Hz total** per harmonic —
about **9× narrower** than the Q=50 notch and **2.4× narrower** than Q=200.

| Method | Effective width / harmonic | Beta preserved |
|--------|--------------------------|----------------|
| Spectrum Fit (2 Hz) | 2.0 Hz | 59% |
| Comb Notch Q=50 | 0.39 Hz | **91%** |
| Comb Notch Q=200 | 0.10 Hz | 104% |
| **FFT Spectral Interp** | **0.042 Hz** | **109%** |
| Sinusoidal Regression | ~0 Hz | 108% |

> Values >100% mean a tiny DBS residual or eye/muscle energy inflates that band;
> ICA removes eye and muscle in the next stage.

## Pipeline stages
1. Bandpass filter (1–119 Hz, FIR zero-phase)
2. **DBS removal** — 5-method comparison, then FFT Spectral Interpolation chosen
3. **ICA** — auto-label eye + muscle components, reconstruct
4. Evaluation — PSD, topomaps, quantitative band-power table

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal
import mne
mne.set_log_level('WARNING')

from src.synthetic_eeg import SyntheticEEG
from src.filters import ArtifactFilterFactory

import pathlib
pathlib.Path('../figures').mkdir(exist_ok=True)

print(f'MNE {mne.__version__}  |  NumPy {np.__version__}')

MNE 1.10.2  |  NumPy 2.2.6


## 1 — Generate synthetic signal

In [2]:
SFREQ    = 256.0
DURATION = 120.0
DBS_FREQ = 7.0
SEED     = 42

gen  = SyntheticEEG(sfreq=SFREQ, duration=DURATION, seed=SEED)
data = gen.generate(dbs_freq=DBS_FREQ)

ch_names = data['ch_names']
times    = data['times']
sfreq    = data['sfreq']

print('Signal RMS per component:')
for k in ('brain', 'dbs', 'eyes', 'muscle', 'mixed'):
    print(f'  {k:8s}  {np.sqrt(np.mean(data[k]**2)):6.2f} µV')

Signal RMS per component:
  brain      11.56 µV
  dbs        73.90 µV
  eyes       14.93 µV
  muscle      2.41 µV
  mixed      76.31 µV


In [3]:
fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)
t_show = times < 10
ch_idx = ch_names.index('Fp1')
for ax, lbl, col in zip(axes,
    ['brain', 'dbs', 'eyes', 'muscle', 'mixed'],
    ['steelblue', 'crimson', 'darkorange', 'forestgreen', 'black']):
    ax.plot(times[t_show], data[lbl][ch_idx, t_show], color=col, lw=0.8)
    ax.set_ylabel(f'{lbl}\n(µV)', fontsize=9)
    ax.set_xlim(0, 10)
axes[-1].set_xlabel('Time (s)')
fig.suptitle('Synthetic EEG components — Fp1, first 10 s', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/synth_01_time_domain.png', dpi=150)
plt.show()

In [4]:
# ── Shared helper functions used throughout the notebook ────────────────

def make_raw(which='mixed', bandpass=True):
    r = gen.to_mne_raw(data, which)
    r.apply_proj()
    if bandpass:
        r.filter(l_freq=1.0, h_freq=min(119.0, sfreq/2 - 1),
                 fir_design='firwin', phase='zero', verbose=False)
    return r

def compute_psd_db(raw, fmax=80.0, n_fft=4096):
    obj = raw.compute_psd(method='welch', fmax=fmax, n_fft=n_fft, verbose=False)
    return obj.freqs, 10 * np.log10(obj.get_data().mean(axis=0) + 1e-30)

def get_psd_linear(raw, fmax=80.0, n_fft=4096):
    obj = raw.compute_psd(method='welch', fmax=fmax, n_fft=n_fft, verbose=False)
    return obj.freqs, obj.get_data()   # (n_ch, n_freq) V²/Hz

def band_power_db(raw, lo, hi):
    p = raw.compute_psd(method='welch', fmin=lo, fmax=hi,
                        n_fft=4096, verbose=False).get_data()
    return 10 * np.log10(p.mean() + 1e-30)

def band_preserve_pct(raw_cleaned, raw_brain, lo, hi):
    """% of brain reference band power retained."""
    b = band_power_db(raw_brain, lo, hi)
    c = band_power_db(raw_cleaned, lo, hi)
    return 100 * 10 ** ((c - b) / 10)

def harmonic_atten_db(raw_before, raw_after, h, bw=0.10):
    pb = raw_before.compute_psd(method='welch', fmin=h-bw, fmax=h+bw,
                                n_fft=4096, verbose=False).get_data().mean()
    pa = raw_after.compute_psd(method='welch',  fmin=h-bw, fmax=h+bw,
                               n_fft=4096, verbose=False).get_data().mean()
    return 10 * np.log10((pa + 1e-30) / (pb + 1e-30))

BANDS = [
    ('Delta', 0.5, 4.0),
    ('Theta', 4.0, 8.0),
    ('Alpha', 8.0, 13.0),
    ('Beta',  13.0, 30.0),
]

# Reference raws
raw_brain = make_raw('brain')
raw_cont  = make_raw('mixed')   # bandpassed contaminated signal

print('Reference raws ready.')

Reference raws ready.


## 2 — DBS removal: 7-method comparison

We run every method on the same contaminated signal and score:

- **Brain preservation (%)** — how much of each EEG band survives vs the clean reference.  
  100% = perfect. < 100% = filter destroyed brain signal. > 100% = artifact residual.
- **Mean DBS harmonic attenuation (dB)** — within ±0.10 Hz of the first four harmonics.  
  ⚠ Wide notches *look* better on this metric by attenuating brain energy inside the window.

| # | Method | Key parameter |
|---|--------|---------------|
| 1 | Spectrum Fit | 2 Hz bandwidth/harmonic |
| 2 | Comb Notch Q=50 | −1 dB BW ≈ 0.39 Hz |
| 3 | Comb Notch Q=200 | −1 dB BW ≈ 0.10 Hz |
| 4 | Hampel Freq | 2 Hz rolling-MAD window |
| 5 | Hampel Time | window = 1 DBS period |
| 6 | **FFT Spectral Interp** | **0.042 Hz / harmonic** |
| 7 | Sinusoidal Regression | ~0 Hz (exact-line removal) |

In [5]:
def apply_dbs_filter(method, **kwargs):
    r = raw_cont.copy()
    r.load_data()
    d = r.get_data() * 1e6   # V → µV
    d_clean = ArtifactFilterFactory.process(method, d, sfreq, **kwargs)
    r._data = d_clean * 1e-6
    return r

methods = [
    ('Spectrum Fit (2 Hz)',    'spectrum_fit',         dict(f_target=DBS_FREQ, bandwidth=2.0)),
    ('Comb Notch  Q=50',      'comb_notch',            dict(f0=DBS_FREQ, q_factor=50)),
    ('Comb Notch  Q=200',     'comb_notch',            dict(f0=DBS_FREQ, q_factor=200)),
    ('Hampel Freq (2Hz)',     'hampel_freq',            dict(window_hz=2.0, n_sigmas=3.0, attenuation_db=-60.0)),
    ('Hampel Time',           'hampel_time',            dict(window_sec=1/DBS_FREQ, n_sigmas=3.0)),
    ('FFT Spectral Interp',   'fft_spectral_interp',   dict(f_target=DBS_FREQ)),
    ('Sinusoidal Regression', 'sinusoidal_regression', dict(f_target=DBS_FREQ)),
]

cleaned = {}
for name, mth, kw in methods:
    print(f'\nRunning: {name}')
    cleaned[name] = apply_dbs_filter(mth, **kw)

print('\nAll 7 methods done.')


Running: Spectrum Fit (2 Hz)
Applying Spectrum-Fit Multi-Harmonic Removal (f=7.0Hz, bandwidth=2.0Hz, attenuation=-60.0dB)

Running: Comb Notch  Q=50
Applying Comb Filter (f0=7.0Hz, Q=50)

Running: Comb Notch  Q=200
Applying Comb Filter (f0=7.0Hz, Q=200)



Running: Hampel Freq (2Hz)
Running vectorized Freq-Domain Hampel (Allen et al., 2010) - window=2.0Hz, sigmas=3.0, attenuation=-60.0dB


Replaced 2698 frequency domain spikes.

Running: Hampel Time
Running vectorized Time-Domain Hampel (Allen et al., 2010) - window=37 samples, sigmas=3.0, attenuation=1.0x


  Pass 1: Replaced 6546 outlier points.

Running: FFT Spectral Interp
FFT Spectral Interpolation: f₀=7.0 Hz, ±2 bins (41.7 mHz per harmonic), 18 harmonics
  → Effective removal: 41.7 mHz per harmonic (18 harmonics, 750.0 mHz total)

Running: Sinusoidal Regression
Sinusoidal Regression: f₀=7.0 Hz, 18 harmonics, chunk=full

All 7 methods done.


In [6]:
# ── Quantitative comparison table ──────────────────────────────────────
harmonics_to_check = [k * DBS_FREQ for k in range(1, 5)]   # 7, 14, 21, 28 Hz

print(f"{'Method':<28}  {'Delta%':>7} {'Theta%':>7} {'Alpha%':>7} {'Beta%':>7}  {'Atten dB':>9}")
print('─' * 72)

metrics = {}
for name, _, _ in methods:
    rc = cleaned[name]
    band_pcts = [band_preserve_pct(rc, raw_brain, lo, hi) for _, lo, hi in BANDS]
    atten = np.mean([harmonic_atten_db(raw_cont, rc, h) for h in harmonics_to_check])
    metrics[name] = dict(zip([b for b,_,_ in BANDS], band_pcts), atten=atten)
    print(f"{name:<28}  "
          f"{band_pcts[0]:>7.1f} {band_pcts[1]:>7.1f} "
          f"{band_pcts[2]:>7.1f} {band_pcts[3]:>7.1f}  {atten:>9.1f}")

print()
print('100% = perfect brain preservation')
print('<100% = filter removed brain signal along with DBS')
print('>100% = artifact residual (eye/muscle) inflates the band — removed by ICA later')

Method                         Delta%  Theta%  Alpha%   Beta%   Atten dB
────────────────────────────────────────────────────────────────────────
Spectrum Fit (2 Hz)             182.3    80.2    90.2    58.6      -52.9


Comb Notch  Q=50                182.3   137.0   100.8    90.7      -31.7


Comb Notch  Q=200               182.3   144.1   101.5   104.0      -19.4
Hampel Freq (2Hz)               166.0   128.7    76.6   101.7      -15.8


Hampel Time                     169.5   382.8    99.2   190.4       -1.1


FFT Spectral Interp             182.3   144.2   101.6   109.2      -14.7


Sinusoidal Regression           182.3   143.6   101.6   108.4      -15.6

100% = perfect brain preservation
<100% = filter removed brain signal along with DBS
>100% = artifact residual (eye/muscle) inflates the band — removed by ICA later


In [7]:
# ── PSD comparison plot — focus on 1–30 Hz ─────────────────────────────
palette = {
    'Spectrum Fit (2 Hz)':    ('sienna',      '--',  1.0),
    'Comb Notch  Q=50':       ('crimson',      '-.',  1.0),
    'Comb Notch  Q=200':      ('orchid',       '-.',  1.0),
    'Hampel Freq (2Hz)':      ('peru',         ':',   1.5),
    'Hampel Time':            ('goldenrod',    ':',   1.5),
    'FFT Spectral Interp':    ('steelblue',    '-',   2.0),
    'Sinusoidal Regression':  ('darkcyan',     '-',   1.5),
}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

for ax, (fmin, fmax), title in zip(
    axes,
    [(1, 80), (1, 30)],
    ['Full spectrum  1–80 Hz', 'Zoom  1–30 Hz  (critical region)'],
):
    # Brain reference
    fb, pb = compute_psd_db(raw_brain, fmax=fmax)
    mb = (fb >= fmin) & (fb <= fmax)
    ax.fill_between(fb[mb], pb[mb] - 2, pb[mb] + 2,
                    color='limegreen', alpha=0.15, label='Brain ± 2 dB')
    ax.plot(fb[mb], pb[mb], color='limegreen', lw=1.0, ls='--', label='Brain ref')

    # Contaminated
    fc, pc = compute_psd_db(raw_cont, fmax=fmax)
    mc = (fc >= fmin) & (fc <= fmax)
    ax.plot(fc[mc], pc[mc], color='gray', lw=0.7, alpha=0.5, label='Contaminated')

    # Each method
    for name, _, _ in methods:
        col, ls, lw = palette[name]
        fn, pn = compute_psd_db(cleaned[name], fmax=fmax)
        mn = (fn >= fmin) & (fn <= fmax)
        ax.plot(fn[mn], pn[mn], color=col, ls=ls, lw=lw, label=name)

    # Mark DBS harmonics
    for k in range(1, int(fmax // DBS_FREQ) + 1):
        h = k * DBS_FREQ
        ax.axvline(h, color='orange', lw=0.7, ls=':', alpha=0.6,
                   label='DBS harmonic' if k == 1 else '')

    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD (dB)')
    ax.set_title(title)
    ax.legend(fontsize=7, loc='upper right')

fig.suptitle('DBS removal method comparison — 5 methods vs brain reference',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/synth_02_method_comparison_psd.png', dpi=150)
plt.show()

In [8]:
# ── Bar chart: Beta preservation vs DBS attenuation ────────────────────
method_names = [n for n, _, _ in methods]
beta_pcts  = [metrics[n]['Beta']  for n in method_names]
atten_vals = [abs(metrics[n]['atten']) for n in method_names]
bar_colors = ['sienna','crimson','orchid','peru','goldenrod','steelblue','darkcyan']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

bars1 = ax1.bar(range(len(method_names)), beta_pcts, color=bar_colors, alpha=0.85)
ax1.axhline(100, color='limegreen', lw=1.5, ls='--', label='100% = perfect')
ax1.axhline(90,  color='orange',    lw=1.0, ls=':',  label='90% warning')
ax1.set_xticks(range(len(method_names)))
ax1.set_xticklabels(method_names, rotation=25, ha='right', fontsize=8)
ax1.set_ylabel('Beta preservation (%)')
ax1.set_title('Brain beta preservation (13–30 Hz)\n100% = no brain lost')
ax1.set_ylim(40, 130)
ax1.legend(fontsize=8)
for bar, val in zip(bars1, beta_pcts):
    ax1.text(bar.get_x()+bar.get_width()/2, val+1, f'{val:.0f}%',
             ha='center', va='bottom', fontsize=8, fontweight='bold')

bars2 = ax2.bar(range(len(method_names)), atten_vals, color=bar_colors, alpha=0.85)
ax2.set_xticks(range(len(method_names)))
ax2.set_xticklabels(method_names, rotation=25, ha='right', fontsize=8)
ax2.set_ylabel('Mean DBS attenuation |dB|')
ax2.set_title('Apparent DBS harmonic attenuation\n⚠ Wide methods inflate this by removing brain too')
for bar, val in zip(bars2, atten_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, val+0.3, f'{val:.1f}',
             ha='center', va='bottom', fontsize=8, fontweight='bold')

fig.suptitle('7-method comparison — Brain preservation vs apparent DBS attenuation',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/synth_03_method_comparison_bars.png', dpi=150)
plt.show()

## 3 — Chosen method: FFT Spectral Interpolation

**Why FFT Spectral Interpolation wins for 7 Hz DBS:**

- Effective removal bandwidth: **0.042 Hz** per harmonic (5 FFT bins)
- Brain theta (4–8 Hz) is a broad oscillation; DBS at 7 Hz is a single spectral line →
  removing one narrow line does not noticeably affect broad-band theta power
- Between harmonics (e.g., 7–14 Hz): zero modification, brain signal untouched
- No filter design, no ringing, no phase distortion outside the replaced bins
- Simple and interpretable: cubic-spline interpolation from flanking bins

In [9]:
raw_dbs_removed = cleaned['FFT Spectral Interp']

# ── Show PSD before / after DBS removal (zoom on theta + harmonics) ────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (fmin, fmax), title in zip(
    axes,
    [(1, 80), (1, 30)],
    ['Full spectrum 1–80 Hz', 'Zoom 1–30 Hz (theta + harmonics)'],
):
    fb, pb = compute_psd_db(raw_brain, fmax=fmax)
    fc, pc = compute_psd_db(raw_cont,  fmax=fmax)
    fd, pd = compute_psd_db(raw_dbs_removed, fmax=fmax)

    mb = (fb >= fmin) & (fb <= fmax)
    ax.plot(fb[mb], pb[mb], color='limegreen', lw=0.9, ls='--', label='Brain ref')
    ax.plot(fc[mb], pc[mb], color='gray',      lw=0.7, alpha=0.6, label='Contaminated')
    ax.plot(fd[mb], pd[mb], color='steelblue', lw=1.8, label='FFT Spectral Interp')

    for k in range(1, int(fmax // DBS_FREQ) + 1):
        h = k * DBS_FREQ
        ax.axvline(h, color='orange', lw=0.8, ls=':', alpha=0.7,
                   label='DBS harmonic' if k == 1 else '')

    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD (dB)')
    ax.set_title(title)
    ax.legend(fontsize=8)

fig.suptitle('FFT Spectral Interpolation — before vs after DBS removal',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/synth_04_chosen_dbs_removal.png', dpi=150)
plt.show()

In [10]:
# ── Theta-band zoom: does 7 Hz removal affect broad theta? ─────────────
fig, ax = plt.subplots(figsize=(9, 5))

fb, pb = compute_psd_db(raw_brain,       fmax=15.0, n_fft=8192)
fc, pc = compute_psd_db(raw_cont,        fmax=15.0, n_fft=8192)
fd, pd = compute_psd_db(raw_dbs_removed, fmax=15.0, n_fft=8192)

m = (fb >= 3) & (fb <= 15)
ax.fill_between(fb[m], pb[m] - 1, pb[m] + 1, color='limegreen', alpha=0.2)
ax.plot(fb[m], pb[m], color='limegreen', lw=1.2, ls='--', label='Brain ref')
ax.plot(fc[m], pc[m], color='gray',      lw=0.8, alpha=0.6, label='Contaminated')
ax.plot(fd[m], pd[m], color='steelblue', lw=2.0, label='FFT Spectral Interp')

# Mark theta band
ax.axvspan(4, 8, alpha=0.07, color='purple', label='Theta band (4–8 Hz)')
ax.axvline(DBS_FREQ, color='orange', lw=1.5, ls=':', label=f'DBS {DBS_FREQ} Hz')
ax.axvline(14, color='orange', lw=1.0, ls=':', alpha=0.5)

ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('PSD (dB)')
ax.set_title('Theta-band preservation after FFT Spectral Interpolation\n'
             'DBS spike at 7 Hz removed; surrounding theta spectrum intact')
ax.legend(fontsize=9)
ax.set_xlim(3, 15)
plt.tight_layout()
plt.savefig('../figures/synth_05_theta_zoom.png', dpi=150)
plt.show()

## 4 — ICA: remove eye & muscle artifacts

### Classifier criteria (conservative — err on the side of keeping brain)

**Eye components** — both conditions required:
1. Largest |weight| in the mixing matrix is on **Fp1 or Fp2** (only channels unambiguously eye-proximal)
2. Source PSD: low-frequency content (1–15 Hz) > 60 % of 1–80 Hz total — eye artifacts are slow

**Muscle components** — both conditions required:
1. Largest |weight| is on a **temporal or outer-frontal** channel (T3/T4/T5/T6/F7/F8)
2. Source PSD: high-frequency content (30–80 Hz) > 55 % of 1–80 Hz total

The double-condition rule prevents misclassifying genuine alpha/beta components as eye artifacts
and prevents misclassifying broadband brain components as muscle.

In [11]:
N_COMPONENTS = 15

ica = mne.preprocessing.ICA(
    n_components=N_COMPONENTS,
    method='fastica',
    max_iter=800,
    random_state=42,
)
ica.fit(raw_dbs_removed, verbose=False)
print(f'ICA fitted: {N_COMPONENTS} components on {len(ch_names)} EEG channels')

ICA fitted: 15 components on 19 EEG channels


In [12]:
def classify_ica_components(ica, raw, sfreq):
    """
    Conservative two-condition classifier for eye and muscle ICA components.

    Eye
    ---
    Condition 1 (topographic): argmax |mixing column| ∈ {Fp1, Fp2}
    Condition 2 (spectral):    LF power (1–15 Hz) / total (1–80 Hz) > 0.60

    Muscle
    ------
    Condition 1 (topographic): argmax |mixing column| ∈ {T3,T4,T5,T6,F7,F8}
    Condition 2 (spectral):    HF power (30–80 Hz) / total (1–80 Hz) > 0.55
    """
    mixing  = ica.get_components()             # (n_ch, n_components)
    sources = ica.get_sources(raw).get_data()  # (n_components, n_samples)

    ch_lower = [c.lower() for c in raw.ch_names]
    fp_idx     = [i for i, c in enumerate(ch_lower) if c in ('fp1', 'fp2')]
    muscle_idx = [i for i, c in enumerate(ch_lower)
                  if c in ('t3', 't4', 't5', 't6', 'f7', 'f8')]

    eye_comps    = []
    muscle_comps = []

    for ic in range(ica.n_components_):
        col    = np.abs(mixing[:, ic])
        top_ch = int(np.argmax(col))

        f_s, psd_s = signal.welch(sources[ic], fs=sfreq, nperseg=512)
        p_all = np.trapz(psd_s[(f_s>=1)&(f_s<=80)], f_s[(f_s>=1)&(f_s<=80)]) + 1e-30
        lf_r  = np.trapz(psd_s[(f_s>=1)&(f_s<=15)], f_s[(f_s>=1)&(f_s<=15)]) / p_all
        hf_r  = np.trapz(psd_s[(f_s>=30)&(f_s<=80)],f_s[(f_s>=30)&(f_s<=80)]) / p_all

        if top_ch in fp_idx and lf_r > 0.60:
            eye_comps.append(ic)
        elif top_ch in muscle_idx and hf_r > 0.55:
            muscle_comps.append(ic)

    return eye_comps, muscle_comps


eye_comps, muscle_comps = classify_ica_components(ica, raw_dbs_removed, sfreq)
all_bad = sorted(set(eye_comps + muscle_comps))

print(f'Eye components    : {eye_comps}')
print(f'Muscle components : {muscle_comps}')
print(f'Total to exclude  : {all_bad}  ({len(all_bad)} / {N_COMPONENTS})')

Eye components    : [0, 1]
Muscle components : []
Total to exclude  : [0, 1]  (2 / 15)


In [13]:
fig = ica.plot_components(
    picks=range(N_COMPONENTS), show=False,
    title='ICA topomaps  (red border = to be removed)',
)
for ax in fig.axes:
    t = ax.get_title()
    if t.startswith('ICA'):
        ic_num = int(t.replace('ICA', ''))
        if ic_num in all_bad:
            for spine in ax.spines.values():
                spine.set_edgecolor('crimson')
                spine.set_linewidth(2.5)
plt.savefig('../figures/synth_06_ica_topomaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
if all_bad:
    fig_src = ica.plot_sources(
        raw_dbs_removed, picks=all_bad, show=False,
        title='Artifact ICA sources (first 10 s)', start=0, stop=10,
    )
    plt.savefig('../figures/synth_07_ica_sources.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('No artifact components detected.')

In [15]:
ica.exclude = all_bad
raw_clean = raw_dbs_removed.copy()
ica.apply(raw_clean, verbose=False)
print(f'ICA applied — removed {len(all_bad)} component(s): {all_bad}')

ICA applied — removed 2 component(s): [0, 1]


## 5 — Final evaluation

In [16]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (fmin, fmax), title in zip(
    axes,
    [(1, 80), (1, 30)],
    ['Full spectrum 1–80 Hz', 'Zoom 1–30 Hz'],
):
    fb, pb = compute_psd_db(raw_brain,       fmax=fmax)
    fc, pc = compute_psd_db(raw_cont,        fmax=fmax)
    fd, pd = compute_psd_db(raw_dbs_removed, fmax=fmax)
    fl, pl = compute_psd_db(raw_clean,       fmax=fmax)

    m = (fb >= fmin) & (fb <= fmax)
    ax.fill_between(fb[m], pb[m]-1.5, pb[m]+1.5, color='limegreen', alpha=0.15)
    ax.plot(fb[m], pb[m], color='limegreen', lw=0.9, ls='--', label='Brain ref')
    ax.plot(fc[m], pc[m], color='gray',      lw=0.7, alpha=0.5, label='Contaminated')
    ax.plot(fd[m], pd[m], color='steelblue', lw=1.2, alpha=0.8, label='DBS removed (FFT interp)')
    ax.plot(fl[m], pl[m], color='darkblue',  lw=2.0, label='Final clean (+ ICA)')

    if fmax <= 30:
        for k in range(1, 5):
            h = k * DBS_FREQ
            if h <= fmax:
                ax.axvline(h, color='orange', lw=0.8, ls=':', alpha=0.7)

    ax.set_xlabel('Frequency (Hz)'); ax.set_ylabel('PSD (dB)')
    ax.set_title(title); ax.legend(fontsize=8)

fig.suptitle('All pipeline stages — PSD comparison', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/synth_08_final_psd.png', dpi=150)
plt.show()

In [17]:
def topomap_band(raw, lo, hi, ax, title):
    obj   = raw.compute_psd(method='welch', fmin=lo, fmax=hi, n_fft=2048, verbose=False)
    bp    = obj.get_data().mean(axis=1)
    bp_db = 10 * np.log10(bp + 1e-30)
    mne.viz.plot_topomap(
        bp_db, raw.info, axes=ax, show=False, cmap='RdBu_r',
        vlim=(np.percentile(bp_db, 5), np.percentile(bp_db, 95)),
    )
    ax.set_title(title, fontsize=8)

fig, axes = plt.subplots(2, 4, figsize=(15, 7))
stages = [
    (raw_cont,        'Contaminated'),
    (raw_dbs_removed, 'DBS removed'),
    (raw_clean,       'Final clean'),
    (raw_brain,       'Brain ref'),
]
for col_i, (raw_s, label) in enumerate(stages):
    topomap_band(raw_s, 4,  8,  axes[0, col_i], f'{label}\nTheta (4–8 Hz)')
    topomap_band(raw_s, 8, 13,  axes[1, col_i], f'{label}\nAlpha (8–13 Hz)')

fig.suptitle('Band power topomaps — all stages vs brain reference',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../figures/synth_09_topomaps.png', dpi=150)
plt.show()

In [18]:
f_cl, p_cl = get_psd_linear(raw_clean)
f_b,  p_b  = get_psd_linear(raw_brain)
f_c,  p_c  = get_psd_linear(raw_cont)
f_d,  p_d  = get_psd_linear(raw_dbs_removed)

def bp_mean(freqs, psd, lo, hi):
    m = (freqs >= lo) & (freqs <= hi)
    return np.trapz(psd[:, m], freqs[m], axis=1).mean()

print('=' * 85)
print('PIPELINE SUMMARY — band power (dB re 1 V²/Hz, mean across 19 channels)')
print('=' * 85)
print(f"{'Band':<22} {'Brain ref':>10} {'Contam.':>10} {'DBS only':>10} {'Final':>10} {'Preserved%':>12}")
print('-' * 85)
for bname, lo, hi in BANDS:
    b_  = bp_mean(f_b,  p_b,  lo, hi)
    c_  = bp_mean(f_c,  p_c,  lo, hi)
    d_  = bp_mean(f_d,  p_d,  lo, hi)
    cl_ = bp_mean(f_cl, p_cl, lo, hi)
    pct = 100 * cl_ / (b_ + 1e-30)
    print(f'{bname:<22} {10*np.log10(b_+1e-30):>10.1f} {10*np.log10(c_+1e-30):>10.1f}'
          f' {10*np.log10(d_+1e-30):>10.1f} {10*np.log10(cl_+1e-30):>10.1f} {pct:>11.1f}%')

print()
print('DBS harmonic attenuation — Contaminated → Final clean:')
for k in range(1, 9):
    h = k * DBS_FREQ
    if h >= sfreq / 2: break
    a = harmonic_atten_db(raw_cont, raw_clean, h)
    print(f'  k={k}  {h:5.1f} Hz  →  {a:+.1f} dB')
print('=' * 85)

PIPELINE SUMMARY — band power (dB re 1 V²/Hz, mean across 19 channels)
Band                    Brain ref    Contam.   DBS only      Final   Preserved%
-------------------------------------------------------------------------------------
Delta                      -106.3     -103.7     -103.7     -106.8        89.0%
Theta                      -110.4     -104.0     -108.9     -110.9        89.7%
Alpha                      -111.1     -111.1     -111.1     -111.6        90.8%
Beta                       -109.9     -106.4     -109.5     -110.0        97.8%

DBS harmonic attenuation — Contaminated → Final clean:
  k=1    7.0 Hz  →  -19.8 dB
  k=2   14.0 Hz  →  -16.6 dB
  k=3   21.0 Hz  →  -14.2 dB
  k=4   28.0 Hz  →  -11.0 dB
  k=5   35.0 Hz  →  -11.3 dB
  k=6   42.0 Hz  →  -10.8 dB
  k=7   49.0 Hz  →  -9.7 dB
  k=8   56.0 Hz  →  -9.8 dB
